# Generate regular-HR to logarithmic-HR matrix
Creates `scatter_to_log_128.pt`. Coordinates are in HR pixels.

In [1]:
import torch
from differentiable_lensing import DifferentiableLensing
THETA_E_ARCSEC=0.75
LR_PIXEL_SCALE_ARCSEC=0.168
HR_PIXEL_SCALE_ARCSEC=0.084
LR_GRID_SHAPE=64
HR_GRID_SHAPE=128
ALPHA_R_LR=THETA_E_ARCSEC/LR_PIXEL_SCALE_ARCSEC
ALPHA_R_HR=THETA_E_ARCSEC/HR_PIXEL_SCALE_ARCSEC
LOG_C=4.5
DEVICE='cpu'
assert HR_GRID_SHAPE==2*LR_GRID_SHAPE
print(ALPHA_R_LR,ALPHA_R_HR)

4.464285714285714 8.928571428571429


In [2]:
lens=DifferentiableLensing(DEVICE,target_resolution=1.0,target_shape=HR_GRID_SHAPE,alpha=None)
lower,upper=-HR_GRID_SHAPE/2.0,HR_GRID_SHAPE/2.0
_,_,regular_x_edges,regular_y_edges=lens.make_center_grid(lower,upper,HR_GRID_SHAPE)
_,_,log_x_edges,log_y_edges=lens.make_log_grid(lower,upper,HR_GRID_SHAPE,c=LOG_C)
grid_fracs=lens.log_grid_crop(log_x_edges,log_y_edges,regular_x_edges,regular_y_edges)
areas=lens.nsq_As(regular_x_edges,regular_y_edges)
M,shape=lens.build_sparse_mapping(grid_fracs,areas,DEVICE)
M=M.coalesce()
assert M.shape==(HR_GRID_SHAPE**2,HR_GRID_SHAPE**2)
assert torch.isfinite(M.values()).all()
torch.save(M,'scatter_to_log_128.pt')
print('saved scatter_to_log_128.pt',M.shape,M._nnz(),shape)

Iterating through axis-oriented grid rows: 100%|██████████| 128/128 [8:45:02<00:00, 246.11s/it]  


saved scatter_to_log_128.pt torch.Size([16384, 16384]) 64516 (128, 128, 128, 128)
